## 🧱 Özel Ortam

---

Bu görevde, derste örnek olarak kullanılan **ızgara tabanlı navigasyon ortamının** birebir aynısını tasarlayıp uygulayacaksınız.

- Ajan ızgaranın bir köşesinde başlar.
- Amaç karşı köşeye ulaşmaktır.
- Izgarada ajanın kaçınması gereken engeller veya "delikler" bulunabilir.

---

### 🎯 Amaçlar

- 📐 **Ortamı Tanımlayın**: ajanın başlangıçtan hedefe doğru hareket ettiği bir ızgara dünyası oluşturun.
- ⚙️ **Ortam Dinamiklerini Uygulayın**: hareket ve ödül ataması kurallarını programlayın (örn. deliklere düşmek için ceza, hedefe ulaşmak için ödül).
- 👁️ **Gözlem ve Eylemleri Ayarlayın**:
    - **Gözlem alanı** → Ajanın algıladığı şeyler (örn. ızgaradaki pozisyonu)  
    - **Eylem alanı** → Ajanın yapabileceği şeyler (örn. yukarı, aşağı, sola, sağa hareket)
- 🖼️ **Render Metodu Ekleyin**: ortamı görselleştirmek için `.render()` fonksiyonu ekleyin — hata ayıklama ve ajanın davranışını anlama için faydalıdır.
- 🧩 **Özel Özellikler Ekleyin**: derste ele alınan engeller, çukurlar veya diğer özellikleri dahil ederek ortamınızı daha dinamik ve gerçekçi hale getirin.

---
Bu görev için ihtiyacımız olan tüm paketleri içe aktararak başlayalım:

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict, Tuple


import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import DQN
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

---

### 🧩 Bölüm 1: Özel Ortam Sınıfı Oluşturma

Bu bölümde, kısmen yazılmış bir sınıfı tamamlayarak özel ortamınızı uygulayacaksınız.  
Yapı, Gymnasium'un ortam oluşturma için standart formatını takip eder.

### 🛠️ Yapacaklarınız

- Bazı temel metotları önceden yazılmış bir şablon sağlanmıştır.  
- `# CODE HERE` yazdığını gördüğünüz her yerde, eksik mantığı doldurmanız gerekir.  
- Her adımda size rehberlik etmesi için satır içi ipuçları verilmiştir.

### 🎯 Hedefiniz

- Sınıfı gerçek bir Gymnasium ortamı gibi davranacak şekilde tamamlayın.  
- Şunları yapmalı:
  - Geçerli bir gözlem ve eylem alanı tanımlamalı  
  - Ajan hareketini ve geçişleri ele almalı  
  - Ödülleri döndürmeli ve `done` bayraklarını uygun şekilde güncellemeli  
  - Çalışan bir `.reset()` ve `.step()` metodu içermeli  
  - İsteğe bağlı olarak görselleştirme için basit bir `.render()` içermeli

🧠 Her metodu anlamak için zaman ayırın — özellikle durum geçişlerinin ve ödüllerin nasıl yönetildiğini. RL ortamları burada canlanır.

📚 Ortam yapısı ve en iyi uygulamalar hakkında ayrıntılı adımlar için resmi [Gymnasium özel ortam kılavuzuna](https://gymnasium.farama.org/introduction/create_custom_env/) başvurun.

In [4]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Optional, Dict, Tuple

class CustomGridEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        self.size = 3  # Size of the square grid
        self.action_space = spaces.Discrete(4)  # 4 olası hareket (sağ, yukarı, sol, aşağı)
        self.agent_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([2, 2], dtype=np.int32)
        self.hole_position = np.array([2, 1], dtype=np.int32)

        self.action_to_direction = {
            0: np.array([0, 1]),   # Move right
            1: np.array([-1, 0]),  # Move up
            2: np.array([0, -1]),  # Move left
            3: np.array([1, 0])    # Move down
        }

        # Sözlük tabanlı gözlem alanı
        self.observation_space = spaces.Dict({
            "agent": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
            "target": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
        })

    def reset(self, seed: Optional[int] = None, options: Optional[Dict] = None) -> Tuple[Dict, Dict]:
        super().reset(seed=seed)

        # Ajanın pozisyonunu başlangıca sıfırla
        self.agent_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([2, 2], dtype=np.int32)

        observation = self._get_obs()
        info = self._get_info()
        return observation, info

    def _get_obs(self):
        return {
            "agent": self.agent_position.copy(),
            "target": self.goal_position.copy()
        }

    def _get_info(self):
        distance = np.sum(np.abs(self.agent_position - self.goal_position))
        return {"distance": distance}

    def step(self, action):
        # 1. Hareketi uygula
        direction = self.action_to_direction[action]
        self.agent_position = np.clip(self.agent_position + direction, 0, self.size - 1)

        # 2. Ödül ve bitiş (terminated) durumlarını kontrol et
        terminated = False
        truncated = False

        if np.array_equal(self.agent_position, self.hole_position):
            reward = -10.0
            terminated = True
        elif np.array_equal(self.agent_position, self.goal_position):
            reward = 10.0
            terminated = True
        else:
            reward = -1.0
            terminated = False

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

    def render(self):
        grid = np.full((self.size, self.size), fill_value=' ')

        agent_x, agent_y = self.agent_position
        goal_x, goal_y = self.goal_position
        hole_x, hole_y = self.hole_position

        grid[agent_x][agent_y] = 'A'
        grid[goal_x][goal_y] = 'G'
        grid[hole_x][hole_y] = 'H'

        print("+---" * self.size + "+")
        for row in grid:
            print("|" + "|".join(f" {cell} " for cell in row) + "|")
            print("+---" * self.size + "+")

👇 Önceki bölümü doğru tamamladığınızı test etmek için aşağıdaki hücreyi çalıştırın. Doğru yaptıysanız, ajanın bir bölümün sonu olan hedefe veya deliğe ulaşana kadar ortamınızda rastgele hareket ettiğini göreceksiniz.

In [5]:
# Create an instance of the custom environment
env = CustomGridEnv()

# Reset the environment to its initial state and get the initial observation and info
obs, info = env.reset()
print("Initial Observation:", obs)  # Display the initial position of the agent and the target
print("Initial Info:", info)  # Display additional information such as the distance from the target

# Loop through a maximum of 100 steps
for _ in range(100):
    action = env.action_space.sample()  # Randomly sample an action from the action space
    obs, reward, done, info, _ = env.step(action)  # Apply the action and get the results
    env.render()  # Render the current state of the environment to visualize the agent's position

    # Print the current state, reward received, whether the episode is done, and any additional info
    print(f"State: {obs}, Reward: {reward}, Done: {done}, Info: {info}")

    # If the episode is finished (agent reached the goal or fell into a hole), exit the loop
    if done:
        break


Initial Observation: {'agent': array([0, 0], dtype=int32), 'target': array([2, 2], dtype=int32)}
Initial Info: {'distance': 4}
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0]), 'target': array([2, 2], dtype=int32)}, Reward: -1.0, Done: False, Info: False
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0]), 'target': array([2, 2], dtype=int32)}, Reward: -1.0, Done: False, Info: False
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0]), 'target': array([2, 2], dtype=int32)}, Reward: -1.0, Done: False, Info: False
+---+---+---+
| A |   |   |
+---+---+---+
|   |   |   |
+---+---+---+
|   | H | G |
+---+---+---+
State: {'agent': array([0, 0]), 'target': array([2, 2], dtype=int32)}, Reward: -1.0, Done: False, Info: False
+---+---+---+
|   | A |   |
+---+---+---+

---
## Bölüm 2: DQN Eğitimi 🤖

Bu bölümde, özel ortamınızı başlatacak ve Stable Baselines3 kütüphanesini kullanarak bir DQN ajanı ile etkileşim için hazırlayacaksınız. Temel görev, ortamınızın kütüphanenin gereksinimleriyle uyumlu olduğundan emin olmaktır, bu da onu uygun şekilde sarmalamayı içerir.

#### 📝 İzlenecek adımlar

1. 🧱 **Özel Ortamı Başlatın**: özel ortam sınıfınızın bir örneğini oluşturun.
2. 🔁 **SB3 Uyumluluğunu Sağlayın**: Stable Baselines3'ten `make_vec_env` fonksiyonunu kullanarak ortamınızı sarın, böylece kütüphanenin vektörleştirilmiş ortam gereksinimleriyle uyumlu hale getirin.
3. ⚙️ **DQN Ajanını Yapılandırın ve Eğitin**: DQN ajanını uygun hiperparametrelerle kurun ve ortamınızda eğitin.
4. 📊 **Eğitim İlerlemesini İzleyin**: Ajanın öğrenme ilerlemesini ve performansını zaman içinde gözlemlemek için günlükleme ve izleme uygulayın.
5. 💾 **Modeli kaydedin**: Eğitim sonrasında modelinizi kaydedin.

In [8]:
import os
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

# --- BÖLÜM 1: Özel Ortam Sınıfımız (Daha önce oluşturmuştuk) ---
class CustomGridEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super().__init__()
        self.size = 3
        self.action_space = spaces.Discrete(4)
        self.agent_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([2, 2], dtype=np.int32)
        self.hole_position = np.array([2, 1], dtype=np.int32)

        self.action_to_direction = {
            0: np.array([0, 1]),   # Sağ
            1: np.array([-1, 0]),  # Yukarı
            2: np.array([0, -1]),  # Sol
            3: np.array([1, 0])    # Aşağı
        }

        self.observation_space = spaces.Dict({
            "agent": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
            "target": spaces.Box(low=0, high=self.size - 1, shape=(2,), dtype=np.int32),
        })

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_position = np.array([0, 0], dtype=np.int32)
        self.goal_position = np.array([2, 2], dtype=np.int32)
        return self._get_obs(), self._get_info()

    def _get_obs(self):
        return {
            "agent": self.agent_position.copy(),
            "target": self.goal_position.copy()
        }

    def _get_info(self):
        distance = np.sum(np.abs(self.agent_position - self.goal_position))
        return {"distance": distance}

    def step(self, action):
        direction = self.action_to_direction[action]
        self.agent_position = np.clip(self.agent_position + direction, 0, self.size - 1)

        terminated, truncated = False, False
        if np.array_equal(self.agent_position, self.hole_position):
            reward = -10.0
            terminated = True
        elif np.array_equal(self.agent_position, self.goal_position):
            reward = 10.0
            terminated = True
        else:
            reward = -1.0

        return self._get_obs(), reward, terminated, truncated, self._get_info()


# --- BÖLÜM 2: DQN Eğitimi Adımları ---

# 1 ve 2. Adım: Özel Ortamı Başlatma ve SB3 Uyumluluğunu Sağlama (Vektörleştirme ve İzleme)
log_dir = "./dqn_grid_logs/"
os.makedirs(log_dir, exist_ok=True)

# make_vec_env ve Monitor kullanarak ortamı SB3 standartlarına uygun hale getiriyoruz
# Not: Sözlük (Dict) tabanlı gözlem uzayları için SB3'ün 'MultiInputPolicy' mimarisini kullanmalıyız.
env = make_vec_env(
    lambda: Monitor(CustomGridEnv(), log_dir),
    n_envs=1
)

# 3. Adım: DQN Ajanını Yapılandırın ve Eğitin
# 'MultiInputPolicy', Dict gözlem uzayını işleyebilmesi için zorunludur.
model = DQN(
    "MultiInputPolicy",
    env,
    verbose=1,
    learning_rate=1e-3,
    buffer_size=10000,
    learning_starts=100,
    batch_size=32,
    gamma=0.99,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05
)

print("--- DQN Eğitimi Başlıyor ---")
# Ortamımız çok küçük olduğu için 10.000 adım (timestep) öğrenmek için fazlasıyla yeterlidir.
model.learn(total_timesteps=30000)
print("--- Eğitim Tamamlandı ---")

# 4. Adım: Eğitim İlerlemesini İzleme (Monitoring)
# Monitor sınıfı otomatik olarak bölümlerin (episode) aldığı toplam ödülleri ve uzunluklarını
# log_dir klasörüne kaydeder. Tensorboard ile görselleştirmek isterseniz terminale şunu yazabilirsiniz:
# tensorboard --logdir ./dqn_grid_logs/

# 5. Adım: Modeli Kaydedin
model_path = "dqn_custom_grid_agent"
model.save(model_path)
print(f"Model başarıyla '{model_path}.zip' olarak kaydedildi.")

Using cpu device
--- DQN Eğitimi Başlıyor ---
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 10.8     |
|    ep_rew_mean      | -19.8    |
|    exploration_rate | 0.986    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 5459     |
|    time_elapsed     | 0        |
|    total_timesteps  | 43       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 14.5     |
|    ep_rew_mean      | -18.5    |
|    exploration_rate | 0.963    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1170     |
|    time_elapsed     | 0        |
|    total_timesteps  | 116      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.664    |
|    n_updates        | 3        |
----------------------------------
----------------------------------
| rollout

---
## 🎮 Bölüm 3: Eğitilmiş modelinizi kullanın

Eğitilmiş DQN modelinizi yükleyecek ve ortamda karar vermek için kullanacaksınız, başlangıç noktasından hedefe delikleri kaçınarak nasıl gittiğini gözlemleyeceksiniz. Bu sadece eğitimin etkinliğini doğrulamanıza izin vermeyecek, aynı zamanda ajanın karar verme sürecini görsel olarak yorumlamanıza da olanak sağlayacaktır.

### İzlenecek adımlar 📝
1. 💾 **Eğitilmiş Modeli Yükleyin**: Eğitim aşamasında kaydedilen DQN modelini alın.
2. 🔄 **Ortamı Sıfırlayın**: Navigasyon görevini sıfırdan başlatmak için ortamı başlatın.
3. 🧠 **Navigasyon Simülasyonunu Çalıştırın**: Modeli kullanarak ortamın durumlarına dayalı eylemleri tahmin edin ve ajanın attığı her adımı görselleştirin.
4. 👀 **Her Adımı Görselleştirin**: Ajanın pozisyonunu, hedefi ve herhangi bir engeli veya deliği gösteren ızgaranın basit bir görselleştirmesini uygulayın.
5. 📝 **Ajan Davranışını Analiz Edin**: Ajanın hedefe ulaşma yeteneğini gözlemleyin ve not edin ve delikleri ne kadar etkili bir şekilde kaçındığını görün.

Kodun bir kısmı zaten orada ve `# CODE HERE` yorumunu gördüğünüz her yerde kod doldurmanız gerekiyor. Başlamanız için bazı ipuçları bulacaksınız.

In [9]:
import numpy as np
from stable_baselines3 import DQN

# 1. Adım: Eğitilmiş Modeli Yükleyin
# Kaydettiğimiz model dosyasını yüklüyoruz
model = DQN.load("dqn_custom_grid_agent")

# 2. Adım: Ortamı Sıfırlayın
# Vektörleştirilmiş ortam (make_vec_env) kullandığımız için reset() bize (obs, info) yerine sadece obs dönebilir
# veya tuple dönebilir. Vektör ortam standardında obs döndürür.
obs = env.reset()

for _ in range(200):  # Maksimum 200 adım döngüsü
    # 3. Adım: Modeli kullanarak bir sonraki eylemi tahmin edin
    # Vektörleştirilmiş ortamda predict, toplu (batch) sonuç döner; deterministic=True en iyi eylemi seçtirir.
    action, _states = model.predict(obs, deterministic=True)

    # Eylemi ortama uygula (Vektör ortam step fonksiyonu: obs, rewards, dones, infos döner)
    obs, rewards, dones, infos = env.step(action)

    # Vektör ortam olduğu için gözlemler bir boyutlu dizi içinde gelir,
    # koordinatları doğru okumak için [0] indisini kullanıyoruz (şablonunuzdaki mantıkla uyumlu).
    agent_x, agent_y = obs['agent'][0][0], obs['agent'][0][1]
    goal_x, goal_y = obs['target'][0][0], obs['target'][0][1]
    hole_x, hole_y = 2, 1  # Çukurun sabit konumu

    # Görselleştirme için 3x3 ızgara oluştur
    grid = np.full((3, 3), fill_value=' ')
    grid[agent_x][agent_y] = 'A'  # Ajan
    grid[goal_x][goal_y] = 'G'  # Hedef
    grid[hole_x][hole_y] = 'H'  # Çukur

    # Durum, ödül, bitti bilgisi ve ek bilgileri yazdır
    done = dones[0] # Vektör yapısından tekil bitti bayrağını alıyoruz
    print(f"State: {obs}, Reward: {rewards[0]}, Done: {done}, Info: {infos[0]}")

    # Izgarayı konsola çizdir
    print("+---" * 3 + "+")
    for row in grid:
        print("|" + "|".join(f" {cell} " for cell in row) + "|")
        print("+---" * 3 + "+")

    print("\n" + "-"*20 + "\n")

    # Eğer oyun bittiyse (hedefe ulaştı veya çukura düştü), döngüyü kırabilir veya ortamı sıfırlayabilirsiniz
    if done:
        print("Bölüm sona erdi! Ortam yeniden başlatılıyor...")
        obs = env.reset()
        # İsteğe bağlı olarak yeni bölüm için kısa bir duraklama koyabilirsiniz

State: OrderedDict({'agent': array([[1, 0]], dtype=int32), 'target': array([[2, 2]], dtype=int32)}), Reward: -1.0, Done: False, Info: {'distance': 3, 'TimeLimit.truncated': False}
+---+---+---+
|   |   |   |
+---+---+---+
| A |   |   |
+---+---+---+
|   | H | G |
+---+---+---+

--------------------

State: OrderedDict({'agent': array([[1, 1]], dtype=int32), 'target': array([[2, 2]], dtype=int32)}), Reward: -1.0, Done: False, Info: {'distance': 2, 'TimeLimit.truncated': False}
+---+---+---+
|   |   |   |
+---+---+---+
|   | A |   |
+---+---+---+
|   | H | G |
+---+---+---+

--------------------

State: OrderedDict({'agent': array([[1, 2]], dtype=int32), 'target': array([[2, 2]], dtype=int32)}), Reward: -1.0, Done: False, Info: {'distance': 1, 'TimeLimit.truncated': False}
+---+---+---+
|   |   |   |
+---+---+---+
|   |   | A |
+---+---+---+
|   | H | G |
+---+---+---+

--------------------

State: OrderedDict({'agent': array([[0, 0]], dtype=int32), 'target': array([[2, 2]], dtype=int32)

🧠 Artık ajanınızın ortamda hareket ettiğini görmelisiniz.

Eğer ajan **belirli eylemleri tekrarlayarak takılıp kalırsa** veya hedefe ulaşamazsa, muhtemelen **yeterince uzun eğitilmediği** içindir.

Eğitim adım sayısını artırmayı deneyin ve modeli yeniden eğitin — daha uzun eğitim genellikle ajanın daha iyi bir politika öğrenmesine yardımcı olur.